# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata (as an object, not a dictionary)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and basic info.

To ensure compliance with FAIR principles, all entities will be referenced by their `@id`.

In [ ]:
# List record sets and fields with their @id
record_sets_info = []
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"  RecordSet @id: {record_set.id}, Name: {record_set.name}")
    field_ids = [field.id for field in record_set.fields]
    print(f"    Fields: {field_ids}")
    record_sets_info.append({
        'id': record_set.id,
        'name': record_set.name,
        'fields': field_ids
    })

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

Below, we create a dictionary of DataFrames indexed by record set `@id`.

In [ ]:
# Extract data from each record set
record_sets_ids = [r['id'] for r in record_sets_info]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet {record_set_id}: {df.columns.tolist()}")
    if not df.empty:
        print(df.head())

# For demonstration, select the first available record set
selected_record_set_id = record_sets_ids[0] if record_sets_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

All fields are referenced using their `@id`, as per FAIR² and Croissant conventions.

In [ ]:
# Identify numeric fields in the selected record set
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_fields}")
    
    # For demonstration, select an example numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()  # Use mean as a threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Group by a possible categorical field (e.g., the second column, if available)
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Use matplotlib for simple visualizations, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualize the distribution of the numeric field, if exists
if selected_record_set_id and numeric_fields:
    field = numeric_field_id
    plt.figure(figsize=(8, 6))
    sns.histplot(df[field], bins=10, kde=True)
    plt.title(f"Distribution of {field} (@id)")
    plt.xlabel(f"{field} (@id)")
    plt.ylabel("Count")
    plt.show()
    
    # If grouped_df was created, visualize group means
    if 'grouped_df' in locals():
        grouped_df = grouped_df.reset_index()
        plt.figure(figsize=(8, 6))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(f"{group_field_id} (@id)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset includes diverse clinicopathological and molecular variables for second primary colorectal cancer in cancer survivors.
- Entities within the dataset are referenced using their `@id`, supporting traceable and reproducible research.
- Through `mlcroissant`, tabular data can be loaded, filtered, normalized, and visualized for further clinical data science analyses.
- For advanced modeling or analysis, expand this notebook by integrating additional fields, more complex filtering, and customized visualizations.